
**Tablas**

samples.tpcds_sf1.customer

samples.tpcds_sf1.item

samples.tpcds_sf1.store

Escribir tres consultas a modo reporte que:
Tengan uno o mas campos calculados, que usen por lo menos una función de Databricks.
Filtren por alguno(s) de los campos calculados.
Apliquen un ordenamiento para mostrar los resultados de forma mas
coherente.

Poner en comentarios qué hace la consulta y por qué agrega valor vs la
tabla original. Se valorará creatividad, cláusulas adicionales y que los
resultados sean correctos.


In [0]:
%sql
  -- La idea de la consulta es visualizar sobre las tiendas activas el tax percentage, al mismo tiempo que se visualiza los años operativos y el numero de empleados

  select
    s_store_id,
    s_store_name,
    s_city,
    s_manager,
    s_number_employees,
    s_rec_start_date,
    -- operative years
    round(datediff(current_date(), s_rec_start_date) / 365.25, 1) as years_operating,
    -- tax analysis 
    s_tax_precentage,
    round(avg(s_tax_precentage) over (partition by s_city), 4) as city_avg_tax,
    round(s_tax_precentage - avg(s_tax_precentage) over (partition by s_city), 4) as tax_diff_vs_city,
    case
      when s_tax_precentage > avg(s_tax_precentage) over (partition by s_city) then 'tax higher than city avg'
      when s_tax_precentage < avg(s_tax_precentage) over (partition by s_city) then 'tax lower than city avg'
      else 'tax equal to city avg'
    end as tax_performance
  from samples.tpcds_sf1.store
  where s_tax_precentage is not null
    and s_floor_space is not null
    and s_number_employees is not null
    and s_city is not null
    and s_rec_end_date is null
    and s_rec_start_date is not null

In [0]:
%sql
-- La idea de la consulta es visualizar los items que mejor margen de beneficio tienen respecto del promedio de ganacia de su categoria permitiendo ver cuales son los items que mas revenue generan por unidad, teniendo como margen inferior un profit del 20%

SELECT DISTINCT
  i_item_id,
  i_brand,
  i_category,
  i_class,
  i_current_price,
  i_wholesale_cost,
  profit_margin_pct,
  profit_per_unit,
  markup_pct,
  years_in_catalog,
  
  -- category average profit margin
  ROUND(AVG(profit_margin_pct) OVER (PARTITION BY i_category), 2) AS category_avg_margin_pct,
  ROUND(profit_margin_pct - AVG(profit_margin_pct) OVER (PARTITION BY i_category), 2) AS margin_diff_vs_category,
  
  -- product classification based on profit margin
  CASE
    WHEN profit_margin_pct > AVG(profit_margin_pct) OVER (PARTITION BY i_category) + 10 
      THEN 'Premium Margin Product'
    WHEN profit_margin_pct < AVG(profit_margin_pct) OVER (PARTITION BY i_category) - 10 
      THEN 'Low Margin Product'
    ELSE 'Standard Margin Product'
  END AS margin_category

FROM (
  -- Subquery calc profit and antiquity
  SELECT
    i_item_id,
    i_brand,
    i_category,
    i_class,
    i_current_price,
    i_wholesale_cost,
    ROUND((i_current_price - i_wholesale_cost) / i_current_price * 100, 2) AS profit_margin_pct,
    ROUND(i_current_price - i_wholesale_cost, 2) AS profit_per_unit,
    ROUND((i_current_price - i_wholesale_cost) / i_wholesale_cost * 100, 2) AS markup_pct,
    ROUND(DATEDIFF(CURRENT_DATE(), i_rec_start_date) / 365.25, 1) AS years_in_catalog
  
  FROM samples.tpcds_sf1.item
  
  WHERE i_rec_end_date IS NULL
    AND i_current_price IS NOT NULL
    AND i_wholesale_cost IS NOT NULL
    AND i_current_price > 0
    AND i_wholesale_cost > 0
    AND i_category IS NOT NULL
    AND i_rec_start_date IS NOT NULL
) AS item_metrics

-- only products that have a profit margin above 20%
WHERE profit_margin_pct > 20

ORDER BY 
  i_category,
  profit_margin_pct DESC,
  i_current_price DESC
LIMIT 10

In [0]:
%sql
-- La idea de la consulta es visualizar de manera rapida la cantidad de clientes por cada segmento de edad, asi com otambien cuantos de ellos dentro del segmento son clientes importantes o favoritos y cuantos son regulares
SELECT DISTINCT
  age_segment,
  min_age,
  max_age,
  customers_in_segment,
  preferred_customers_in_segment,
  regular_customers_in_segment,
  
  -- percentage of vip/favorite customers in the segment
  ROUND(preferred_customers_in_segment * 100.0 / customers_in_segment, 2) AS preferred_pct

FROM (
  -- Subquery calc customers per segment
  SELECT
    age_segment,
    min_age,
    max_age,
    
    COUNT(*) OVER (PARTITION BY age_segment) AS customers_in_segment,
    
    COUNT(CASE WHEN c_preferred_cust_flag = 'Y' THEN 1 END) 
      OVER (PARTITION BY age_segment) AS preferred_customers_in_segment,
    
    COUNT(CASE WHEN c_preferred_cust_flag = 'N' THEN 1 END) 
      OVER (PARTITION BY age_segment) AS regular_customers_in_segment
  
  FROM (
    -- Subquery calc age segments
    SELECT
      c_customer_id,
      c_preferred_cust_flag,
      YEAR(CURRENT_DATE()) - c_birth_year AS current_age,
      
      
      CASE
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 18 AND 25 THEN '18-25 Young Adults'
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 26 AND 35 THEN '26-35 Millennials'
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 36 AND 45 THEN '36-45 Mid Career'
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 46 AND 60 THEN '46-60 Established'
        WHEN YEAR(CURRENT_DATE()) - c_birth_year > 60 THEN '60+ Seniors'
        ELSE 'Unknown'
      END AS age_segment,
      
      -- min and max range values
      CASE
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 18 AND 25 THEN 18
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 26 AND 35 THEN 26
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 36 AND 45 THEN 36
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 46 AND 60 THEN 46
        WHEN YEAR(CURRENT_DATE()) - c_birth_year > 60 THEN 60
      END AS min_age,
      
      CASE
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 18 AND 25 THEN 25
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 26 AND 35 THEN 35
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 36 AND 45 THEN 45
        WHEN YEAR(CURRENT_DATE()) - c_birth_year BETWEEN 46 AND 60 THEN 60
        WHEN YEAR(CURRENT_DATE()) - c_birth_year > 60 THEN 99
      END AS max_age
    
    FROM samples.tpcds_sf1.customer
    
    WHERE c_birth_year IS NOT NULL
      AND c_preferred_cust_flag IS NOT NULL
      AND YEAR(CURRENT_DATE()) - c_birth_year >= 18
      AND YEAR(CURRENT_DATE()) - c_birth_year <= 120
  ) AS customer_ages
  
  WHERE age_segment != 'Unknown'
  
) AS segment_counts

-- only segments with at least 1000 customers
WHERE customers_in_segment >= 1000

ORDER BY 
  customers_in_segment DESC,
  age_segment